# Fine-tune NLLB for English ⇄ Kikamba (Understand Kamba)

Uses only existing public data (see `training/README.md`). About **2 hours** on a free T4 GPU.

**Before running (Kaggle → right panel → Session options):**
- Accelerator: **GPU T4 x2**
- Internet: **On** (needs a phone-verified account)
- Persistence: **Files only** (keeps progress if the session restarts)

Then click **Run All** and keep this tab open. Every step prints progress with a time-left estimate.
Rerunning is safe: finished steps are saved in `/kaggle/working` and skipped.

In [ ]:
# 1. Setup: check the GPU and internet, get the code, install libraries
import os, socket, torch
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU T4 x2"
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet: Session options -> Internet -> On (verify your phone in Kaggle Settings if greyed out)")

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
if not os.path.isdir("understand-kamba"):
    !git clone -q --depth 1 -b training https://github.com/lawmaluki/understand-kamba.git
os.chdir(f"{WORK}/understand-kamba")
!git pull -q
!pip uninstall -y -q torchao
!pip install -q "transformers>=4.45" peft sacrebleu sentencepiece accelerate
print("Ready:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Train and evaluate (about 2 hours). Progress lines appear as it goes; warnings go to train_errors.txt.
# Steps: score the original model (~25 min) -> filter web pairs (~6 min) -> train (~1 h) -> score the fine-tuned model (~25 min)
!CUDA_VISIBLE_DEVICES=0 python training/finetune_nllb.py --model facebook/nllb-200-distilled-1.3B --out nllb-kam-ft 2> train_errors.txt
if not os.path.exists("nllb-kam-ft/eval_report.json"):
    print("Training stopped before the end. Last lines of the error log:")
    !grep -v "it/s\]" train_errors.txt | tail -n 25

In [ ]:
# 3. Results: original vs fine-tuned, and some translations from the held-out stories
import json
if os.path.exists("nllb-kam-ft/eval_report.json"):
    r = json.load(open("nllb-kam-ft/eval_report.json"))
    print(f"{'test set / direction':28} {'original':>9} {'fine-tuned':>11}")
    for k, v in r["finetuned"].items():
        b = r.get("base", {}).get(k)
        print(f"{k:28} {b if b is not None else '-':>9} {v:>11}  {'' if b is None else f'{v - b:+.2f}'}")
    d = json.load(open("nllb-kam-ft/test_outputs.json", encoding="utf-8"))
    print()
    for (kam, eng), hyp in list(zip(d["tests"]["stories"], d["outputs"]["stories/kam_to_en"]))[:10]:
        print("KAMBA    :", kam); print("REFERENCE:", eng); print("MODEL    :", hyp); print()
else:
    print("No results yet: step 2 did not finish (see its output).")

## 4. (Optional) Upload the model to your Hugging Face account

Only if the fine-tuned model is better on **both** test sets. Set `UPLOAD = True`, run the cell, and paste a
fresh **write** token when asked. The model repository is created **private**.

In [ ]:
UPLOAD = False
if UPLOAD:
    from huggingface_hub import HfApi, login
    login()
    repo = "lawmaluki/nllb-200-1.3B-kamba-ft"
    HfApi().create_repo(repo, private=True, exist_ok=True)
    HfApi().upload_folder(folder_path="nllb-kam-ft", repo_id=repo,
                          allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"],
                          ignore_patterns=["checkpoints/*", "test_outputs.json", "opus_kept_*.json"])
    print("uploaded: https://huggingface.co/" + repo)